# 86 — Analyst Memo Generator

## Goal
Collect **market news**, **filings**, and **internal notes**,
then generate a concise analyst memo with **proper citations**.

## What You'll Learn
| Concept | Detail |
|---------|--------|
| **Multi-source retrieval** | Combine news, filings, notes |
| **Citation tracking** | Track which source supports each claim |
| **Structured memo** | Executive summary, analysis, recommendations |
| **LangChain RAG** | Retrieval-augmented generation with sources |

## Stack
- `langchain-ollama` — LLM
- `chromadb` — vector store for sources
- Ollama `qwen3.5:9b`

In [ ]:
import os, warnings, json
os.environ["USE_TF"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore")

from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, SystemMessage
import chromadb

llm = ChatOllama(model="qwen3.5:9b", temperature=0)
print("All imports OK")

## Step 1 — Source Materials

Three types of sources, each with metadata for citation tracking.

In [ ]:
# Source materials with citation metadata
SOURCES = [
    # Market News
    {"id": "NEWS-001", "type": "news", "source": "Reuters", "date": "2024-11-10",
     "title": "Global Cloud Infrastructure Spending Surges 22% in Q3",
     "text": "Global cloud infrastructure spending reached $84 billion in Q3 2024, up 22% YoY. AWS maintained market lead at 31%, followed by Azure at 25% and GCP at 12%. Enterprise adoption of AI workloads drove demand, with GPU instances seeing 3x growth."},
    
    {"id": "NEWS-002", "type": "news", "source": "Bloomberg", "date": "2024-11-08",
     "title": "Enterprise AI Adoption Reaches Inflection Point",
     "text": "A Bloomberg survey of 500 enterprises shows 67% now have AI in production, up from 39% a year ago. Data infrastructure and MLOps tools are the fastest-growing categories. Budget allocation for AI infrastructure doubled in 2024."},
    
    {"id": "NEWS-003", "type": "news", "source": "TechCrunch", "date": "2024-11-05",
     "text": "DataBricks raised $10B at $62B valuation, the largest private AI infrastructure round ever. Snowflake responded by acquiring two ML startups. The data platform war is intensifying.",
     "title": "DataBricks Mega-Round Signals Data Platform Consolidation"},
    
    # Filings / Reports
    {"id": "FILING-001", "type": "filing", "source": "CloudCorp 10-Q", "date": "2024-10-28",
     "title": "CloudCorp Q3 2024 Quarterly Report",
     "text": "Revenue: $3.2B (+18% YoY). Net income: $320M. Gross margin: 72%. Customer count: 4,200 (+15%). Net revenue retention: 128%. R&D spend increased 25% focusing on AI features. Guidance: Q4 revenue $3.4-3.5B."},
    
    {"id": "FILING-002", "type": "filing", "source": "CloudCorp 10-Q", "date": "2024-10-28",
     "title": "CloudCorp Risk Factors Update",
     "text": "Key risks: increasing competition from hyperscalers, customer concentration (top 10 = 22% of revenue), talent retention in AI/ML roles, and regulatory uncertainty around AI governance in EU markets."},
    
    # Internal Notes
    {"id": "NOTE-001", "type": "internal", "source": "PM Team Standup", "date": "2024-11-12",
     "title": "Product Roadmap Update",
     "text": "AI feature launch on track for January. Beta feedback is positive — 85% of beta users report productivity gains. Key concern: GPU cost optimization needed; current unit economics are thin on AI workloads."},
    
    {"id": "NOTE-002", "type": "internal", "source": "Sales Leadership", "date": "2024-11-11",
     "title": "Enterprise Pipeline Review",
     "text": "Pipeline is strong: $45M in qualified opportunities for Q1 2025. Three $2M+ deals in late stage. Competitive pressure from Databricks increasing — they are offering aggressive discounts. Win rate stable at 38%."}
]

# Build vector store
chroma_client = chromadb.Client()
memo_collection = chroma_client.create_collection("analyst_sources")

memo_collection.add(
    documents=[s["text"] for s in SOURCES],
    metadatas=[{"id": s["id"], "type": s["type"], "source": s["source"], "date": s["date"], "title": s["title"]} for s in SOURCES],
    ids=[s["id"] for s in SOURCES]
)

print(f"Loaded {memo_collection.count()} sources:")
for s in SOURCES:
    print(f"  [{s['type'].upper():8}] {s['id']}: {s['title']}")

## Step 2 — Retrieve Relevant Sources by Topic

In [ ]:
def retrieve_sources(topic: str, n: int = 5) -> list:
    """Retrieve relevant sources for a topic."""
    results = memo_collection.query(query_texts=[topic], n_results=n)
    sources = []
    for i in range(len(results["documents"][0])):
        sources.append({
            "text": results["documents"][0][i],
            "metadata": results["metadatas"][0][i],
            "distance": results["distances"][0][i]
        })
    return sources

# Test retrieval
topic = "CloudCorp competitive position and AI strategy"
sources = retrieve_sources(topic)
print(f"Retrieved {len(sources)} sources for topic: '{topic}'\n")
for s in sources:
    print(f"  [{s['metadata']['id']}] {s['metadata']['title']} (dist={s['distance']:.2f})")

## Step 3 — Generate Analyst Memo with Citations

In [ ]:
def generate_memo(topic: str) -> str:
    """Generate an analyst memo with citations."""
    # Step 1: Retrieve all relevant sources
    sources = retrieve_sources(topic, n=6)
    
    # Format sources with citation keys
    source_text = ""
    for s in sources:
        meta = s["metadata"]
        source_text += f"\n[{meta['id']}] ({meta['source']}, {meta['date']}) {meta['title']}\n{s['text']}\n"
    
    # Step 2: Generate memo
    msg = llm.invoke([
        SystemMessage(content="""You are a senior equity research analyst writing an internal memo.
Write a concise analyst memo with these sections:

1. EXECUTIVE SUMMARY (2-3 sentences)
2. KEY FINDINGS (3-5 bullet points)
3. MARKET CONTEXT (1 paragraph)
4. COMPANY ANALYSIS (1-2 paragraphs)
5. RISKS (bullet points)
6. RECOMMENDATION (1-2 sentences)

IMPORTANT: Cite sources using [SOURCE_ID] format after each claim.
Example: Revenue grew 18% YoY [FILING-001].

Be factual, concise, and data-driven. No thinking or reasoning tags."""),
        HumanMessage(content=f"Topic: {topic}\n\nSources:\n{source_text}")
    ])
    
    memo = msg.content
    if "<think>" in memo:
        memo = memo.split("</think>")[-1].strip()
    
    return memo

print("Memo generator ready")

In [ ]:
# Generate the memo
topic = "CloudCorp Q3 performance, competitive position, and AI strategy outlook"

print("=" * 70)
print("ANALYST MEMO")
print(f"Topic: {topic}")
print(f"Date: 2024-11-13")
print("=" * 70)

memo = generate_memo(topic)
print(memo)

In [ ]:
# Citation verification — check which sources were actually cited
import re

cited_ids = set(re.findall(r'\[([A-Z]+-\d+)\]', memo))
all_ids = {s["id"] for s in SOURCES}

print("\n📚 Citation Report")
print("=" * 40)
print(f"Sources available: {len(all_ids)}")
print(f"Sources cited: {len(cited_ids)}")
print(f"\nCited: {', '.join(sorted(cited_ids))}")
print(f"Not cited: {', '.join(sorted(all_ids - cited_ids))}")

## 🧠 Key Concepts Recap

| Concept | Implementation |
|---------|---------------|
| **Multi-source retrieval** | News + filings + notes in ChromaDB |
| **Citation tracking** | `[SOURCE_ID]` format in prompt instructions |
| **Structured output** | Memo with defined sections |
| **Citation verification** | Regex extraction to verify sources used |

## Pipeline
```
Sources (news, filings, notes) → ChromaDB
        ↓
Topic Query → Retrieve top-K sources
        ↓
LLM: Generate memo with [CITATION] format
        ↓
Citation verification (regex check)
```

## Extending This Project
- Add real RSS/API news feeds
- SEC EDGAR integration for filings
- Sentiment scoring per source
- Hallucination detection (verify claims against sources)